# TESS: run a demo, then change its settings

## Start here: Run All → three search iterations → your progress plot

**Run All performs paid LLM calls and real GPU training through your project's saved script.** Before starting, initialize this external project, prepare the data, and launch Jupyter from a terminal with the required API keys exported. Use the exp checkout's kernel. Do not paste keys into cells.

The first section below is a complete short search demo: **save its configuration → review the saved files → invoke its script → plot score versus iteration**. It uses three search iterations, not three training batches. LLM work can dominate runtime; this is a small search, not a promised seconds-long example. A finite or valid score is not guaranteed.

Set `RUN_QUICK_DEMO=False` to skip API/GPU execution. Quick A still saves input files in your external project. Re-running the same completed demo reuses its records and redraws the figure without another API call. To run changed settings, choose a new `DEMO_NAME`; previous results are retained.

After the plot, the advanced sections teach individual parameter edits and task splits. Their save switches remain opt-in. **The quick demo ends at the validation progress plot; it never automatically chooses a model for an independent final test.**


**What are we doing?** Give a model a star's brightness measurements over time (a *light curve*), and ask it to predict the star's rotation frequency in cycles/day. The agent proposes model code and training settings. Our script trains and evaluates that model.

**What can you change later?** After the quick demo, four optional exercises teach separate changes:

1. **Search longer:** change iterations and epochs.
2. **Use less data in Trial:** choose independent training/validation fractions; keep Formal full-sized.
3. **Fit your hardware/time allowance:** change Trial and Formal budgets.
4. **Make a new train/validation split:** keep each star together and create a new task/data pair.

Follow the [tutorial README](https://github.com/yuema137/siderius-exp/tree/main/tutorials/paper) to initialize your external project, stage the data and configure keys first. Open this project's copied notebook. **Run All now includes a real three-iteration demo:** it invokes your saved launch script, then plots the records. Set `RUN_QUICK_DEMO=False` to skip API/GPU execution. The `WRITE_DEMOS` and `MAKE_NEW_SPLIT` switches save the optional advanced examples; they do not control Quick A’s input files.

Quick A saves an experiment JSON and launch script; Quick B invokes that script for you. The advanced route below lets you launch a different saved example from a terminal. You never edit the source repositories.

**First run:** follow Quick A → Quick B → Quick C, or use Run All after setup. Stop at the plot; no second launch is needed.

**Advanced/manual runs:** after saving a different example, go to **Before you run: one experiment, one checklist, one launch command** near the end. It gives the file checklist, actual saved parameters, and the exact terminal command for your selected example.

## Optional advanced route after the quick demo

1. Locate your external project and choose the existing-data or download option.
2. Read the controls, then use Demos 1–3 to change iterations, data fractions or budgets.
3. Save the chosen experiment/script; optionally use Demo 4 to create a new whole-star task/data split.
4. Go to **Before you run**: select ONE saved example, inspect files/parameters, then copy its preview and launch commands into a terminal.
5. Read the resulting search records and validation/Health evidence.

This TESS tutorial runs training plus workflow validation. It does not stage the held-out test set or provide the separate `seal`/final-test path demonstrated in the TIDMAD file-holdout tutorial. Formal is a stage inside the search, not an untouched final test.

Before launching, both the notebook and saved script check the environment. An error lists missing files or key names and how to fix them. Keys must be exported before starting Jupyter; a kernel restart alone cannot import variables added later to a different terminal. The native launcher then verifies source revisions, data integrity and CUDA. Key presence does not guarantee provider access or available credit.

**What counts as a completed demo:** the saved configuration launches through its script, records the real attempts, and produces the progress plot. Health PASS is not required. Hollow invalid points are expected possible outcomes; this tutorial does not tune the model until checks pass.


## First, what do iteration, Trial and Formal mean?

```text
one iteration = propose/implement a candidate model
                → Trial: try training settings and inspect feedback
                → Formal: evaluate the chosen Trial configuration
next iteration = use earlier evidence to propose another candidate
```

This tutorial fixes **two logical rounds per iteration**: a Trial opportunity and a Formal opportunity. `iterations=3` therefore allows up to three such cycles, not three training epochs. An **epoch** is a pass through the selected training curves. `epochs=10` caps training at ten passes per attempt.

Failed proposals or training attempts may be retried, and failure can prevent a Formal result. Three iterations do not guarantee six successful scores. Formal uses the task's **validation** set; it is not an evaluation on the held-out test set. In this fixed workflow, Formal inherits the chosen Trial model/training configuration; its data scopes and resource budgets are separately controlled.

The training objective helps the optimizer learn. The reported **R²** measures predictions against the targets: larger is better, 1 is perfect, 0 matches the validation-set mean predictor, and negative is worse. A finite R² or `completed` status alone does not mean the output passed every scientific check.

In [ ]:
import json
import os
import shlex
import sys
from pathlib import Path

project_env = os.environ.get("TUTORIAL_HOME")
if not project_env:
    raise RuntimeError(
        "Missing TUTORIAL_HOME. In your terminal, export TUTORIAL_HOME=/absolute/path/to/your/tutorial-project, "
        "then restart Jupyter from that terminal. Use the external project created in the README setup step."
    )
PROJECT = Path(project_env).expanduser().resolve()
if not (PROJECT / "project.json").is_file():
    raise RuntimeError(
        f"Missing {PROJECT / 'project.json'}. Point TUTORIAL_HOME at your initialized external project, "
        "or complete the README's create-project command first. Do not use the source repo as your project."
    )
binding = json.loads((PROJECT / "project.json").read_text())
EXP, INFRA = Path(binding["exp_checkout"]), Path(binding["infra_checkout"])
if Path(sys.prefix).resolve() != (EXP / ".venv").resolve():
    raise RuntimeError(
        f"Wrong notebook kernel: {sys.executable}. Select the kernel installed from {EXP / '.venv/bin/python'}. "
        f"If missing, run: cd {shlex.quote(str(EXP))} && uv sync --group dev --group tutorial --frozen; "
        "then register/select that environment's Jupyter kernel as shown in the README."
    )
import importlib.util

missing_modules = [name for name in ("pydantic", "yaml", "numpy", "h5py", "torch", "workflows", "matplotlib")
                   if importlib.util.find_spec(name) is None]
if missing_modules:
    raise RuntimeError(
        f"Missing Python modules: {', '.join(missing_modules)}. "
        f"Run: cd {shlex.quote(str(EXP))} && uv sync --group dev --group tutorial --frozen; then restart this kernel."
    )
sys.dont_write_bytecode = True
os.chdir(EXP)  # import installed tools; edited files always use absolute project paths
from tutorials.paper.project import write_launcher
from tutorials.paper.runner import TutorialExperiment, credential_status
from tutorials.paper.task_view import inspect_task

base = TutorialExperiment.model_validate_json(
    (PROJECT / "experiments/tess-experiment.json").read_text()
)
print("Your editable project:", PROJECT)
print("Your task:", base.composition)
print("Your data:", base.data_dir)
print("Original run output:", base.workspace)

## Quick demo A — save and check the actual inputs

The next cell creates a new experiment JSON and matching shell script **inside your project**, then shows their paths, saved parameter table, data checks and exact launch command. It never overwrites a different experiment under the same name. If you see **STOP**, fix the stated prerequisite before running the search cell.

You can change the numbers below before the first run. `iterations=3` is the outer search count; each iteration can contain Trial/Formal attempts and retries. Five epochs is a ceiling, and the native per-attempt budget may stop earlier. `gpu` and `data_dir` inherit your project's initial experiment; change those saved initial bindings before creating a new demo if necessary.


If this demo already completed, its workspace is expected to exist: the next cell reuses the saved run rather than relaunching it. A suppressed new-launch command is not a failure of result reuse.

With the default `DEMO_NAME="quick-demo-001"`, these are the files to open in JupyterLab’s file browser:

```text
YOUR_PROJECT/
├── tasks/tess/compositions/rotation_regression.yaml  task entry
├── experiments/tess_quick-demo-001.json  saved settings
├── scripts/run-tess_quick-demo-001.sh    launch entrypoint
├── runs/tess_quick-demo-001/             actual run records
└── plots/tess_quick-demo-001/            PNG, SVG and CSV
```

Quick A prints their absolute paths and reads the settings back from disk. Quick B runs this exact script; Quick C reads this run folder.


In [ ]:
from IPython.display import Markdown, display

from tutorials.paper.launch_review import launch_review
from tutorials.paper.quick_demo import prepare_demo, run_demo

DEMO_NAME = "quick-demo-001"  # Use a NEW name when changing saved settings.
RUN_QUICK_DEMO = True  # Run All will invoke the script: paid APIs + GPU work.
QUICK_SETTINGS = {
    "iterations": 3,
    "epochs": 5,
    "trial_minutes": 1.0,
    "formal_minutes": 2.0,
    "vram_gib": 8.0,
    "trial_train_fraction": 1.0,
    "trial_val_fraction": 1.0,
    "formal_train_fraction": 1.0,
    "formal_val_fraction": 1.0,
}
demo = prepare_demo(PROJECT, "tess", name=DEMO_NAME, settings=QUICK_SETTINGS)
display(Markdown(launch_review(demo.experiment, demo.script, task="tess")))

## Quick demo B — run the saved script and wait

`run_demo` invokes **the exact shell script displayed above with `--launch`**. The script remains responsible for key/data/GPU checks and workflow execution; the notebook does not contain training code. Keep the kernel running. This cell prints elapsed time and the external console-log path while the search runs.

If a model fails, the real failure records are retained and plotted. If launch fails before producing records, inspect the log and fix the prerequisite rather than treating an empty graph as success. Kernel interruption stops the child process group; an interrupted run is not silently restarted.


In [ ]:
if RUN_QUICK_DEMO:
    demo_result = run_demo(demo)
    print(demo_result)
else:
    print("Execution skipped. The next cell can plot an existing run.")

## Quick demo C — see what the search actually did

**Read this as a validation learning curve, not a final-test score.** The x-axis is the outer search iteration; points are recorded **Formal** attempts, never substituted Trial scores.

- **Filled circle:** successful attempt with measured Health PASS.
- **Hollow circle:** Health FAIL or a failed attempt. A raw metric may exist even when the accepted score is null; it is shown as invalid, not promoted to a valid result.
- **Dashed line:** Formal results. **Solid line:** current best. **Star:** a new best with Health PASS.
- As in the paper, negative scores appear as triangles at the axis boundary; exact values remain in the CSV. Missing scores and unknown Health evidence remain in the CSV without an extra marker or panel.

As in the paper's three-task progress figure, the line tracks the best **recorded** Formal score, including hollow invalid points. A high line therefore does not prove a healthy model. This tutorial keeps the paper’s plot style and uses iteration rather than elapsed time. Online Health is not the paper's separate retrospective behavioral review.

The cell saves a PNG, SVG and a CSV containing each point's iteration, attempt ID, status and source record. For another run, change **only `PLOT_WORKSPACE`** to its run directory and execute this plotting cell; this does not launch a search. The raw records remain the authority.


In [ ]:
from tutorials.paper.progress import plot_progress

PLOT_WORKSPACE = demo.workspace  # Or Path("/your/project/runs/another_run")
PLOT_OUTPUT = PROJECT / "plots" / PLOT_WORKSPACE.name
if PLOT_WORKSPACE.is_dir():
    progress_figure = plot_progress(
        PLOT_WORKSPACE,
        PLOT_OUTPUT,
        title="TESS",
        expected_iterations=QUICK_SETTINGS["iterations"]
        if PLOT_WORKSPACE == demo.workspace
        else None,
    )
    display(progress_figure)
    import matplotlib.pyplot as plt

    plt.close(progress_figure)
    print("PNG:", PLOT_OUTPUT / "score-versus-iteration.png")
    print("SVG:", PLOT_OUTPUT / "score-versus-iteration.svg")
    print("Point records:", PLOT_OUTPUT / "score-versus-iteration.csv")
else:
    print(
        "No run workspace yet. Run Quick demo B or select an existing PLOT_WORKSPACE."
    )

## Advanced walkthrough — edit individual choices deliberately

The quick demo above is complete once you have inspected its plot and records. The sections below explain each file and control in more detail. Their save cells create separate examples; they do not overwrite or rerun the quick demo.

## Your files: what exists now, and what a save step will create

A **task package is a directory**, and its composition YAML is the entry point. An **experiment is one JSON file** that selects that entry and sets budgets, routing and paths. A **script is one shell file** whose `EXPERIMENT` line names the saved JSON to launch.

```text
YOUR_PROJECT/                                     printed as PROJECT above
├── notebooks/01_tess_tutorial.ipynb               your editable notebook
├── tasks/tess/                                   your complete task-package copy
│   ├── compositions/rotation_regression.yaml     task entry; links the pieces below
│   ├── data/manifests/rotation_identity.csv      curve identities, targets and splits
│   ├── declared/dataset_profile.json            input layout and counts
│   ├── declared/metric_r2.json                  primary metric declaration
│   └── runtime/                                data and scoring implementations
├── experiments/tess-experiment.json              initial saved experiment
├── experiments/less-trial-data.json              CREATED when WRITE_DEMOS=True
├── llm/agents.json                               provider/model choices, never keys
├── advice/human_advice.example.json              inactive NoPrior example
├── scripts/run-tess.sh                           initial experiment's launcher
├── scripts/run-less-trial-data.sh                CREATED when WRITE_DEMOS=True
├── data/tess-data/                               two staged NPZ inputs
└── runs/demo_less_trial_data/                    CREATED by the actual launch
```

`experiment.composition` points to the task's YAML; `experiment.data_dir` points to its matching data; `experiment.workspace` points to future output. Raw arrays do not live in the task package. Editing a Python variable in this notebook changes memory only. Saving a JSON/YAML/CSV changes the files that the script will read.

## Data source A or B: do not download existing machine data again

**A — On the shared RTX 5090 machine**, read the existing `/home/klz/Data/TESS/split`. The helper verifies the two pinned regression Parquet files and converts them into the framework's two NPZ inputs in your project (about 18 MB total). It does not download or copy the raw Parquet files and does not stage the test split. This small format conversion is needed because the current task adapter reads NPZ.

If you already have a verified two-NPZ data directory, you can instead set the saved experiment's `data_dir` directly to that absolute directory and reuse it. Do not repeat preparation when `data/tess-data` already exists.

**B — On your own machine**, use the pinned download/preparation helper. It downloads only the required train/validation Parquet files into your project's `data/raw-tess/`, then writes `data/tess-data/`.

Choose one data source below. The cell prints a terminal command; it does not execute a download or preparation. Both options feed the same task and experiment format.

In [ ]:
DATA_ENTRY = "existing"  # use "download" on a machine without the shared data
if DATA_ENTRY == "existing":
    preparation = [
        str(EXP / ".venv/bin/python"),
        "-B",
        "-m",
        "tutorials.paper.data_entry",
        "--task",
        "tess",
        "--project",
        str(PROJECT),
        "--source",
        "/home/klz/Data/TESS/split",
    ]
elif DATA_ENTRY == "download":
    preparation = [
        str(EXP / ".venv/bin/python"),
        "-B",
        "-m",
        "tutorials.paper.prepare_tess",
        "--raw-dir",
        str(PROJECT / "data/raw-tess"),
        "--data-dir",
        str(PROJECT / "data/tess-data"),
    ]
else:
    raise ValueError("Choose existing or download")
print("Run once from the exp checkout:", shlex.join(preparation))
print("Current experiment data_dir:", base.data_dir)
print("Prepared files:", [p.name for p in base.data_dir.glob("*.npz")])

## Your controls: change these values in the demo cells

Fractions are between **0.01 and 1.0**: `0.25` means 25%, not 25. They apply to an existing split; they do not create a split.

| Setting | What it changes | Example |
|---|---|---|
| `iterations` | Number of propose → Trial → Formal cycles | `1` → `3` |
| `epochs` | Training-pass ceiling per attempt, both rounds | `1` → `10` |
| `trial_train_fraction` | Fraction of the existing training population available in Trial | `1.0` → `0.25` |
| `trial_val_fraction` | Fraction of existing validation used for Trial feedback | `1.0` → `0.5` |
| `formal_train_fraction` | Fraction of existing training used in Formal | keep `1.0` for full training |
| `formal_val_fraction` | Fraction of existing validation used for Formal scoring | keep `1.0` for comparable full-validation scores |
| `trial_minutes`, `formal_minutes` | Training allowance **per attempt**, independently | `2 / 5` → `3 / 10` minutes |
| `vram_gib` | Shared model VRAM budget unless overridden below | `8` GiB |
| `trial_vram_gib`, `formal_vram_gib` | Optional round-specific override of `vram_gib` | `6 / 8` GiB |

The initialized project fixes both Trial fractions at 1.0 so its data usage is explicit. Setting a Trial fraction to `None` delegates that choice to the agent; it does **not** mean zero. Formal fractions remain operator-owned.

TESS selects whole stars until it reaches the requested number of curves, so the exact count may round upward. The lower-level `train_portion` parameter is not exposed here: this TESS adapter uses the selected training scope for each epoch; it is not a second data-size control for this task.

Changing the split itself is different. That changes which stars belong to train or validation; see demo 4.

In [ ]:
# Helper: keep paths/names unique and validate all edited values.
# Every demo below starts from the same original experiment, not the previous demo.
def variant(name, **changes):
    values = base.model_dump(mode="json")
    values.update(
        workspace=str(PROJECT / "runs" / name),
        run_name=name,
        **changes,
    )
    return TutorialExperiment.model_validate(values)


baseline = variant("demo_baseline", trial_train_fraction=1.0, trial_val_fraction=1.0)
task = inspect_task(baseline)
print(
    "Current task population:",
    len(task.train),
    "training curves;",
    len(task.val),
    "validation curves",
)

## Demo 1 — “I want the agent to search longer”

Change `iterations` to 3 and `epochs` to 10. The agent can explore three candidate cycles, and each training attempt can use up to ten epochs. It can still stop earlier because of time or other execution rules.

This can increase LLM calls, training time and cost. It does not change the task, metric, or train/validation membership. If you only want more training for each candidate, change `epochs` and leave `iterations=1`.

In [ ]:
search_more = variant(
    "demo_search_more",
    iterations=3,
    epochs=10,
    trial_train_fraction=1.0,
    trial_val_fraction=1.0,
)
print("Before:", baseline.iterations, "iteration;", baseline.epochs, "epoch ceiling")
print(
    "After: ",
    search_more.iterations,
    "iterations;",
    search_more.epochs,
    "epoch ceiling",
)
print("Run output:", search_more.workspace)

## Demo 2 — “Try less data first, then use all data for Formal”

Here Trial uses **25% of train** and **50% of validation**. Formal uses **100% of train** and **100% of validation**. The train/validation split stays exactly the same.

For the released TESS task, 25% of 3,338 curves is a target of about 835 curves; because a star's observations stay together, the task's seed-42 example actually selects **843**. Half of 442 validation curves selects **222** in that example. The next cell asks the real task implementation, rather than assuming these counts for every variant.

This is not “25% train and 75% validation.” The two fractions have different denominators. Preview seed 42 illustrates one selection; actual attempt seeds may select different stars and slightly different counts.

Less data can make a Trial cheaper, but very small scopes can fail runtime calibration. If that happens, increase the fractions; do not disable the checks. Keeping `formal_val_fraction=1.0` avoids changing the Formal evaluation population between candidates. Reducing it creates a different evaluation protocol.

In [ ]:
less_trial_data = variant(
    "demo_less_trial_data",
    trial_train_fraction=0.25,
    trial_val_fraction=0.5,
    formal_train_fraction=1.0,
    formal_val_fraction=1.0,
)
scope_example = inspect_task(less_trial_data)
for name, count in scope_example.example_counts.items():
    print(f"{name}: {count} curves (illustrative seed 42)")

## Demo 3 — “Fit the run to my GPU and time allowance”

This example gives each Trial training attempt 3 minutes and 6 GiB, and each Formal attempt 10 minutes and 8 GiB. Change those four numbers to see the resulting settings. Keep both VRAM limits below your GPU's physical memory, leaving room for the driver/runtime.

These are execution budgets, **not** a reservation of GPU memory, and the minute settings are **not** a deadline for the complete script. Proposal generation, code generation, inference, scoring and retries add work outside a single training allowance. This teaching entrypoint has no overall wall-clock or dollar cap.

The installed launcher supports one RTX 5090 or H100. Selecting H100 does not transform a 5090 into an H100; the script checks the actual card and a CUDA allocation. AMD/Intel are unsupported. Smaller budgets may reject a candidate instead of making it successfully run faster.

In [ ]:
resource_limits = variant(
    "demo_resource_limits",
    trial_train_fraction=1.0,
    trial_val_fraction=1.0,
    trial_minutes=3,
    formal_minutes=10,
    trial_vram_gib=6,
    formal_vram_gib=8,
)
print(
    "Trial:",
    resource_limits.trial_minutes,
    "minutes;",
    resource_limits.trial_vram_gib,
    "GiB",
)
print(
    "Formal:",
    resource_limits.formal_minutes,
    "minutes;",
    resource_limits.formal_vram_gib,
    "GiB",
)

## Save the examples; choose what to run at the launch checklist below

Set `WRITE_DEMOS = True` once to create each experiment and its launch script. This writes only inside your external project and refuses existing filenames. It does not start a run. You can also edit a saved experiment JSON directly before launching.

Your script reads its saved experiment file; changing a notebook variable without saving does not change a run. To repeat an experiment, give it a new `workspace` and `run_name` and save a new file. Never use the project root as a run workspace.

Credentials belong in the environment of the launching terminal. They do not belong in notebook cells, scripts or `llm/agents.json`. That JSON selects models/providers only. The check below reports names/presence, not values; key presence is not an API-access test. Configure keys before starting Jupyter, or restart its server from the configured terminal.

After saving, these are the exact connections:

| Script in `scripts/` | Reads JSON in `experiments/` | Task selected by that JSON |
|---|---|---|
| `run-tess.sh` | `tess-experiment.json` | Initial task; unchanged by the demo cells |
| `run-baseline.sh` | `baseline.json` | `tasks/tess/` |
| `run-search-more.sh` | `search-more.json` | `tasks/tess/` |
| `run-less-trial-data.sh` | `less-trial-data.json` | `tasks/tess/` |
| `run-resource-limits.sh` | `resource-limits.json` | `tasks/tess/` |
| `run-new-split.sh` (demo 4) | `new-split.json` | `tasks/tess-split-seed42/` |

To run your edited example, choose its row's script. The initial `run-tess.sh` does not automatically switch to whichever notebook cell you edited most recently.

Do not choose a launch command here. Finish any task/split edits first, then use **Before you run: one experiment, one checklist, one launch command** below.

In [ ]:
WRITE_DEMOS = False


def save_demo(name, experiment):
    config = PROJECT / "experiments" / f"{name}.json"
    script = PROJECT / "scripts" / f"run-{name}.sh"
    if config.exists() or script.exists():
        raise ValueError(
            "Choose new filenames; existing experiments/scripts are not overwritten"
        )
    with config.open("x") as stream:
        stream.write(experiment.model_dump_json(indent=2))
    write_launcher(script, config, INFRA)
    return script


demos = {
    "baseline": baseline,
    "search-more": search_more,
    "less-trial-data": less_trial_data,
    "resource-limits": resource_limits,
}
for name, experiment in demos.items():
    if WRITE_DEMOS:
        save_demo(name, experiment)
    script = PROJECT / "scripts" / f"run-{name}.sh"
    print(name, "→", experiment.workspace)
    print("Preview after saving:", shlex.join(["bash", str(script)]))

key_status = credential_status(base.llm_config)
print("Required key presence:", key_status)
if not all(key_status.values()):
    print(
        "WARNING: export missing keys in the launching terminal. Launch will refuse until configured."
    )

## Demo 4 — “I want a new train/validation split” (advanced)

**This creates a new task, not just a cheaper Trial.** Earlier demos selected fewer curves *within* each existing split. Here we pool the released train+validation curves and assign whole stars to new train/validation groups.

The example asks for **20% of stars in validation**, with seed 42. Since stars have different numbers of light curves, 20% of stars need not be exactly 20% of curves. The preview shows both counts.

Why group by star? If one star's sector-1 curve is in training and its sector-2 curve is in validation, the validation set is no longer independent at the star level. All observations of the same Gaia star must go to the same side.

The helper creates these files together:

1. A new task copy with an updated `data/manifests/rotation_identity.csv` (membership and labels).
2. Two new NPZ archives whose keys match that manifest; flux and target values are preserved.
3. Updated task population metadata and a receipt with the seed, parent hashes and new identity.
4. A new experiment selecting **both** the new task composition and the new data directory, plus its own script/output path.

The 403 held-out test curves are never read or reassigned. The original task/data remain unchanged. This changes validation membership, so do not compare its R² directly with the paper's scores or reuse an old model trained on stars now in validation. Start a fresh run and qualify the new task. This is a data-preparation example, not a scientific validation of a better split.

When `MAKE_NEW_SPLIT=True`, the complete saved structure is:

```text
tasks/tess-split-seed42/                         new task-package directory
├── compositions/rotation_regression.yaml       new experiment's task entry
├── data/manifests/rotation_identity.csv         NEW membership and target rows
└── declared/dataset_profile.json               updated population metadata
data/tess-split-seed42/
├── tess_rotation_train.npz                     curves matching the new train rows
└── tess_rotation_val.npz                       curves matching the new val rows
experiments/new-split.json                      points to BOTH new task and new data
scripts/run-new-split.sh                        points to experiments/new-split.json
runs/demo_new_split/                            created later by that script
```

Open the CSV in Jupyter to inspect membership, the experiment JSON to verify both paths, and the script to verify its `EXPERIMENT` line. Then use `run-new-split.sh`, not the original `run-tess.sh`. Changing only the CSV without regenerating matching archives would be inconsistent; the helper saves the task/data pair together.

In [ ]:
from tutorials.paper.resplit import SplitChoice, choose_split, read_rows, resplit_task

source_task = base.composition.parents[1]
choice = SplitChoice(validation_fraction=0.20, seed=42)
split_preview = choose_split(read_rows(source_task), choice)
print("Before:", len(task.train), "train curves;", len(task.val), "validation curves")
print("After (preview only):")
print(
    "  Train:",
    split_preview.train_stars,
    "stars;",
    split_preview.train_curves,
    "curves",
)
print(
    "  Validation:",
    split_preview.validation_stars,
    "stars;",
    split_preview.validation_curves,
    "curves",
)

In [ ]:
MAKE_NEW_SPLIT = False  # requires the original staged train/validation data
new_task = PROJECT / "tasks/tess-split-seed42"
new_data = PROJECT / "data/tess-split-seed42"
if MAKE_NEW_SPLIT:
    resplit_task(base, new_task, new_data, choice)
    resplit_experiment = variant(
        "demo_new_split",
        composition=str(new_task / "compositions/rotation_regression.yaml"),
        data_dir=str(new_data),
        trial_train_fraction=1.0,
        trial_val_fraction=1.0,
    )
    new_scope = inspect_task(resplit_experiment)
    assert new_scope.fingerprint != task.fingerprint
    assert not (
        {key.split(":")[0] for key in new_scope.train}
        & {key.split(":")[0] for key in new_scope.val}
    )
    new_script = save_demo("new-split", resplit_experiment)
    print("Created task/data pair; star overlap is zero; fingerprint changed.")
    print("Saved launcher:", new_script)
    print("Select new-split in the launch checklist below.")
else:
    print("No split files written. Enable MAKE_NEW_SPLIT once to create this variant.")

## What belongs to the task, and what belongs to the experiment?

Now that you have changed both, the boundary is concrete:

| What you want to change | File you edit in YOUR project | What remains the same |
|---|---|---|
| Iterations, epochs, fractions, time/VRAM | `experiments/<name>.json` | Task split and metric |
| Model/provider | `llm/agents.json`, selected by experiment `llm_config` | Task science |
| Train/validation membership | A new task manifest **and matching NPZ archives**, as in demo 4 | Other tasks and their data |
| Input shape, target, metric or scientific checks | Task declarations **and** corresponding implementation/tests | Requires new task qualification |
| Which saved experiment to launch | Your script's `EXPERIMENT` path | Installed framework source |

Each experiment's `composition` points to the task's `compositions/rotation_regression.yaml`, which connects declarations to their implementations. The task owns the meaning of the data and score; the experiment owns the conditions under which you run it. The notebook edits and explains; the shell script checks and runs.

All examples use the existing fixed workflow and **NoPrior** information treatment. Human advice, Data Analysis and literature review are disabled here. The copied `advice/human_advice.example.json` is inactive; `advice_file` must stay `null`. Enabling advice requires a separately defined treatment, not just editing that example file.

Your files stay under `notebooks/`, `tasks/`, `experiments/`, `llm/`, `scripts/`, `advice/`, `data/` and `runs/` in the external project. The README gives the full directory map and installation commands.

## Before you run: one experiment, one checklist, one launch command

For a first run with edited Trial proportions, select **`less-trial-data`**. It means exactly:

```text
scripts/run-less-trial-data.sh                 ← the file bash executes
  └─ experiments/less-trial-data.json           ← fractions, budgets, paths
       ├─ tasks/tess/compositions/rotation_regression.yaml
       │    └─ data/manifests/rotation_identity.csv (inside that task package)
       ├─ llm/agents.json                     ← provider/model routing, no keys
       ├─ data/tess-data/                     ← prepared NPZ inputs
       └─ runs/demo_less_trial_data/          ← NEW results, created by launch
```

The JSON's actual paths win if you edited them. The report below prints those actual paths; this diagram is only the example.

| Select `RUN_EXAMPLE` | Saved experiment in `experiments/` | Executed file in `scripts/` |
|---|---|---|
| `initial` | `tess-experiment.json` | `run-tess.sh` |
| `baseline` | `baseline.json` | `run-baseline.sh` |
| `search-more` | `search-more.json` | `run-search-more.sh` |
| `less-trial-data` | `less-trial-data.json` | `run-less-trial-data.sh` |
| `resource-limits` | `resource-limits.json` | `run-resource-limits.sh` |
| `new-split` | `new-split.json` | `run-new-split.sh` |

Select `new-split` only after Demo 4 saved its new task and data pair. The initial `run-tess.sh` does not switch to your edited demo automatically.


**Do this once for the ONE experiment you intend to run:**

1. Save your edited experiment and its matching script using the earlier save cell (or use the initial experiment).
2. Select its name in the next cell. Run that cell to reload the actual files from disk.
3. In output **A**, open the listed files in JupyterLab's file browser and check their connections. In **B**, check every saved parameter. In **C**, check input files, a fresh output path and required key names.
4. If something is wrong, edit the named file, **save it**, and rerun this cell. Do not rely on values left in notebook memory.
5. In output **D**, copy the **preview line** into a terminal. After checking preview and exporting keys there, copy the **launch line** into that same terminal. That is the command which starts the experiment.

The cell below only reviews files and prints commands. It does not start training, call an LLM, download data, or change your saved configuration. Missing data or an already-used output workspace produces a **STOP** instead of a launch command.

**Credentials are separate from the files in your project.** If you use a trusted shell environment file outside the project/repositories, make it readable only by you (`chmod 600 /absolute/path/to/credentials.env`). In the launching terminal, use:

```bash
set -a
source /absolute/path/to/credentials.env   # replace with YOUR existing trusted file
set +a
```

That file contains the required variable assignments, for example `OPENAI_API_KEY=...`; never paste real values into this notebook. Alternatively export keys through your managed secret system. Preview shows presence only; launch checks required keys again. Neither a notebook variable nor a file that has not been sourced configures the launching terminal.


In [ ]:
from IPython.display import Markdown, display

from tutorials.paper.launch_review import launch_review

RUN_EXAMPLE = "less-trial-data"  # Choose ONE name from the table above.
if RUN_EXAMPLE not in {
    "initial",
    "baseline",
    "search-more",
    "less-trial-data",
    "resource-limits",
    "new-split",
}:
    raise ValueError("Choose RUN_EXAMPLE from the table above")
config_name = "tess-experiment" if RUN_EXAMPLE == "initial" else RUN_EXAMPLE
script_name = "tess" if RUN_EXAMPLE == "initial" else RUN_EXAMPLE
selected_config = PROJECT / "experiments" / f"{config_name}.json"
selected_script = PROJECT / "scripts" / f"run-{script_name}.sh"
if selected_config.is_file() and selected_script.is_file():
    display(Markdown(launch_review(selected_config, selected_script, task="tess")))
else:
    print(
        "Optional advanced example is not saved. The quick demo above is independent; enable the advanced save cell only if you want this additional example."
    )

## Read the result: did it run, and what did it actually learn?

The script writes `runs/<name>.tutorial.json` as a launch receipt. Node records, trained models and calibration artifacts live inside `runs/<name>/`. Console output appears in the launching terminal unless you redirect it to a project-local log.

The RTX 5090 smoke run completed in about 6½ minutes with 12 API calls. Its first tiny Trial failed calibration; a retry used full scopes and batch size 4. Formal R² was about -0.0625 after one epoch, and the observational prediction-dispersion check failed because outputs were almost constant. The result was **diagnostic**, not paper reproduction. That duration and recovery are observations, not guarantees.

Inspect the score, Trial/Formal role, scientific checks and authority together. An observational check can fail while the run still completes. A failed attempt with no recorded round is shown as `Unspecified`, not guessed to be Formal.

In [ ]:
# This reads completed runs only. Before a launch the list is empty.
result_files = sorted((PROJECT / "runs").rglob("run_output_*.json"))
for path in result_files:
    print(path)
if not result_files:
    print("No results yet. Save a demo and launch its script from a terminal.")

In [ ]:
# Optional: choose one run_output_*.json from the list above.
SELECTED_RECORD = None  # e.g. experiment.workspace / ".../run_output_....json"
if SELECTED_RECORD is not None:
    import pandas as pd
    from agent.schemas.hyperparam_tuning import HyperparamTuningOutput

    output = HyperparamTuningOutput.model_validate_json(
        Path(SELECTED_RECORD).read_text()
    )
    display(
        pd.DataFrame(
            [
                {
                    "round": record.logical_round,
                    "role": ("Trial" if record.is_trial else "Formal")
                    if record.logical_round is not None
                    else "Unspecified",
                    "status": record.status,
                    "metric": record.metric_result.metric_id
                    if record.metric_result
                    else None,
                    "score": record.metric_result.scalar
                    if record.metric_result
                    else None,
                    "health_checks": [
                        {
                            "name": gate.gate_name,
                            "passed": gate.check_passed,
                            "role": gate.gate_role,
                            "action": gate.resolved_action,
                        }
                        for gate in record.health_gate_results
                    ],
                    "authority": record.scientific_authority,
                }
                for record in output.all_records
            ]
        )
    )
else:
    print(
        "Select a saved tuner record after a run; no scores are invented before then."
    )

## Limits of these demos

These examples teach configuration and execution. They do not reproduce the full six-hour paper campaign, guarantee model quality, or qualify a new scientific split. The paper's historical supervisor is a separate entrypoint described in the README.

Generated code runs in native framework subprocesses under your user account; this is not an OS security sandbox. Keep unrelated data and secrets outside run directories. Small fractions, short budgets and low epoch limits may cause rejected attempts. Use the recorded reason to decide what to change rather than disabling runtime or scientific checks.